# Assignment: Advanced Prompt Engineering for Financial Investment Calculations

## Objective
This notebook solves a five-year investment problem with varying annual returns, a fixed annual fee, and tax on gains. It demonstrates three prompting techniques:

1. **Chain of Thought (CoT):** calculate the balance year by year and log every step.
2. **Self-Consistency Chain of Thought (SC-CoT):** solve the same problem using two independent calculation methods and compare the results.
3. **Few-Shot Prompting:** simulate a simple model that uses worked examples to handle a new investment problem.
4. **Conclusion:** summarize the results, strengths, and limitations of each technique.

> **Important note:** These are Python simulations of prompting strategies. The notebook does not call or train a real large language model. The `SimpleModel` class stores examples and uses a deterministic calculation procedure to produce a reproducible answer.

## Financial assumptions
Because the assignment does not provide numerical inputs, this notebook uses an illustrative scenario. Replace the values in the next cell if your instructor has specified different numbers.

- Initial investment: **$10,000**
- Annual returns over five years: **8%, -3%, 12%, 6%, and 10%**
- Fixed annual fee: **$100**, deducted each year
- Tax rate: **20%** on a **positive net annual gain after the fee**
- If the net annual gain is zero or negative, no tax is charged for that year.
- The fee is deducted once per year after applying that year's return.
- Currency values are rounded to cents for display; calculations retain full precision until the end of each year.

This explicitly defines the calculation rules so all methods can be compared fairly.

In [1]:
from dataclasses import dataclass
from typing import Any, Dict, List
import json
import re

# Shared example scenario (change these values if your assignment provides its own).
INVESTMENT_INPUT = {
    "initial_investment": 10_000.00,
    "annual_returns": [0.08, -0.03, 0.12, 0.06, 0.10],
    "annual_fee": 100.00,
    "tax_rate": 0.20,
}

def validate_inputs(prompt: Dict[str, Any]) -> None:
    """Validate the structured financial inputs before calculation."""
    required = {"initial_investment", "annual_returns", "annual_fee", "tax_rate"}
    missing = required - set(prompt)
    if missing:
        raise ValueError(f"Missing required fields: {sorted(missing)}")
    if prompt["initial_investment"] < 0:
        raise ValueError("Initial investment cannot be negative.")
    if prompt["annual_fee"] < 0:
        raise ValueError("Annual fee cannot be negative.")
    if not 0 <= prompt["tax_rate"] <= 1:
        raise ValueError("Tax rate must be between 0 and 1.")
    if not prompt["annual_returns"]:
        raise ValueError("Provide at least one annual return.")
    if any(r < -1 for r in prompt["annual_returns"]):
        raise ValueError("An annual return cannot be below -100%.")

print("Investment inputs:")
print(json.dumps(INVESTMENT_INPUT, indent=2))

Investment inputs:
{
  "initial_investment": 10000.0,
  "annual_returns": [
    0.08,
    -0.03,
    0.12,
    0.06,
    0.1
  ],
  "annual_fee": 100.0,
  "tax_rate": 0.2
}


## 1. Chain of Thought (CoT) Prompting

The function `chain_of_thought(prompt)` calculates the investment balance one year at a time. For each year, it logs:

1. Opening balance
2. Return rate and investment gain/loss
3. Balance after return
4. Annual fee
5. Net gain used for tax
6. Tax charged
7. Closing balance

The calculation rule for each year is:

\[
\text{investment gain} = \text{opening balance} \times \text{return rate}
\]

\[
\text{net taxable gain} = \max(0, \text{investment gain} - \text{annual fee})
\]

\[
\text{tax} = \text{net taxable gain} \times \text{tax rate}
\]

\[
\text{closing balance} = \text{opening balance} + \text{investment gain} - \text{fee} - \text{tax}
\]

The log provides an auditable calculation trail rather than just the final number.

In [2]:
def chain_of_thought(prompt: Dict[str, Any]) -> Dict[str, Any]:
    """Calculate the balance year by year and log the arithmetic for each year."""
    validate_inputs(prompt)
    balance = float(prompt["initial_investment"])
    fee = float(prompt["annual_fee"])
    tax_rate = float(prompt["tax_rate"])
    log: List[Dict[str, Any]] = []

    print("CHAIN OF THOUGHT: YEAR-BY-YEAR INVESTMENT LEDGER")
    print("=" * 68)
    print(f"Initial investment: ${balance:,.2f}")

    for year, rate in enumerate(prompt["annual_returns"], start=1):
        opening_balance = balance
        investment_gain = opening_balance * rate
        after_return = opening_balance + investment_gain
        net_gain_after_fee = investment_gain - fee
        taxable_gain = max(0.0, net_gain_after_fee)
        tax = taxable_gain * tax_rate
        closing_balance = after_return - fee - tax

        entry = {
            "year": year,
            "opening_balance": opening_balance,
            "return_rate": rate,
            "investment_gain": investment_gain,
            "balance_after_return": after_return,
            "annual_fee": fee,
            "net_gain_after_fee": net_gain_after_fee,
            "taxable_gain": taxable_gain,
            "tax": tax,
            "closing_balance": closing_balance,
        }
        log.append(entry)

        print(f"\nYear {year}: return = {rate:.2%}")
        print(f"  Opening balance             : ${opening_balance:,.2f}")
        print(f"  Investment gain/loss        : ${investment_gain:,.2f}")
        print(f"  Balance after return        : ${after_return:,.2f}")
        print(f"  Annual fee                  : ${fee:,.2f}")
        print(f"  Net gain after fee          : ${net_gain_after_fee:,.2f}")
        print(f"  Taxable gain (not below $0) : ${taxable_gain:,.2f}")
        print(f"  Tax                         : ${tax:,.2f}")
        print(f"  Closing balance             : ${closing_balance:,.2f}")
        balance = closing_balance

    print("\nFINAL INVESTMENT VALUE:", f"${balance:,.2f}")
    return {"final_amount": balance, "yearly_log": log}

cot_result = chain_of_thought(INVESTMENT_INPUT)

CHAIN OF THOUGHT: YEAR-BY-YEAR INVESTMENT LEDGER
Initial investment: $10,000.00

Year 1: return = 8.00%
  Opening balance             : $10,000.00
  Investment gain/loss        : $800.00
  Balance after return        : $10,800.00
  Annual fee                  : $100.00
  Net gain after fee          : $700.00
  Taxable gain (not below $0) : $700.00
  Tax                         : $140.00
  Closing balance             : $10,560.00

Year 2: return = -3.00%
  Opening balance             : $10,560.00
  Investment gain/loss        : $-316.80
  Balance after return        : $10,243.20
  Annual fee                  : $100.00
  Net gain after fee          : $-416.80
  Taxable gain (not below $0) : $0.00
  Tax                         : $0.00
  Closing balance             : $10,143.20

Year 3: return = 12.00%
  Opening balance             : $10,143.20
  Investment gain/loss        : $1,217.18
  Balance after return        : $11,360.38
  Annual fee                  : $100.00
  Net gain after fee  

## 2. Self-Consistency Chain of Thought (SC-CoT)

Self-consistency should build on the `chain_of_thought(prompt)` function defined in Section 1. Therefore, **Method A reuses the existing CoT function** rather than duplicating its year-by-year calculation. This keeps the assignment's functions connected and avoids unnecessary repeated code.

- **Path A — Existing CoT function:** call `chain_of_thought(prompt)` and use its detailed yearly ledger and final amount.
- **Path B — Independent recurrence calculation:** calculate the same investment using a separate compact formula, logging each year's closing balance.

For year \(t\), the recurrence is:

\[
B_t = B_{t-1} + B_{t-1}r_t - F - \tau\max(0, B_{t-1}r_t-F)
\]

where \(B_{t-1}\) is the opening balance, \(r_t\) is that year's return, \(F\) is the fixed fee, and \(\tau\) is the tax rate.

The two paths are compared using a small floating-point tolerance. This is a meaningful consistency check because one path uses the detailed CoT ledger and the other uses a separately written recurrence calculation. Both paths must use the same financial assumptions.


In [3]:
def self_consistency_chain_of_thought(prompt: Dict[str, Any]) -> Dict[str, Any]:
    """Reuse the CoT function as Path A and compare it with an independent Path B."""
    validate_inputs(prompt)

    # Path A: reuse the CoT function already defined in Section 1.
    print("SC-CoT PATH A: REUSING chain_of_thought(prompt)")
    cot_path = chain_of_thought(prompt)
    balance_a = cot_path["final_amount"]
    path_a = [row["closing_balance"] for row in cot_path["yearly_log"]]

    # Path B: independently calculate using the recurrence equation.
    initial = float(prompt["initial_investment"])
    fee = float(prompt["annual_fee"])
    tax_rate = float(prompt["tax_rate"])
    returns = prompt["annual_returns"]
    balance_b = initial
    path_b = []

    print("\nSC-CoT PATH B: INDEPENDENT RECURRENCE CALCULATION")
    print(f"Initial investment: ${balance_b:,.2f}")
    for year, rate in enumerate(returns, start=1):
        opening_balance = balance_b
        gain = opening_balance * rate
        taxable_gain = max(0.0, gain - fee)
        tax = taxable_gain * tax_rate
        balance_b = opening_balance + gain - fee - tax
        path_b.append(balance_b)
        print(
            f"Year {year}: open=${opening_balance:,.2f}, return={rate:.2%}, "
            f"gain/loss=${gain:,.2f}, fee=${fee:,.2f}, "
            f"taxable gain=${taxable_gain:,.2f}, tax=${tax:,.2f}, "
            f"close=${balance_b:,.2f}"
        )

    difference = abs(balance_a - balance_b)
    tolerance = 1e-8
    consistent = difference <= tolerance
    print("\nSELF-CONSISTENCY CHECK")
    print(f"Path A (existing CoT) final amount: ${balance_a:,.2f}")
    print(f"Path B (recurrence) final amount : ${balance_b:,.2f}")
    print(f"Absolute difference              : ${difference:.12f}")
    print("Results match:", consistent)

    if not consistent:
        raise ArithmeticError("The CoT and recurrence paths produced different results.")

    return {
        "method_a_final": balance_a,
        "method_b_final": balance_b,
        "absolute_difference": difference,
        "consistent": consistent,
        "method_a_yearly_balances": path_a,
        "method_b_yearly_balances": path_b,
    }

sc_result = self_consistency_chain_of_thought(INVESTMENT_INPUT)


SC-CoT PATH A: REUSING chain_of_thought(prompt)
CHAIN OF THOUGHT: YEAR-BY-YEAR INVESTMENT LEDGER
Initial investment: $10,000.00

Year 1: return = 8.00%
  Opening balance             : $10,000.00
  Investment gain/loss        : $800.00
  Balance after return        : $10,800.00
  Annual fee                  : $100.00
  Net gain after fee          : $700.00
  Taxable gain (not below $0) : $700.00
  Tax                         : $140.00
  Closing balance             : $10,560.00

Year 2: return = -3.00%
  Opening balance             : $10,560.00
  Investment gain/loss        : $-316.80
  Balance after return        : $10,243.20
  Annual fee                  : $100.00
  Net gain after fee          : $-416.80
  Taxable gain (not below $0) : $0.00
  Tax                         : $0.00
  Closing balance             : $10,143.20

Year 3: return = 12.00%
  Opening balance             : $10,143.20
  Investment gain/loss        : $1,217.18
  Balance after return        : $11,360.38
  Annual fee  

## 3. Few-Shot Prompting

Few-shot prompting supplies worked examples before asking the model to solve a new problem. The examples demonstrate the input format and the financial rules.

`SimpleModel` is a **teaching simulation**, not a trained neural network. It stores the examples and uses the same deterministic financial calculator to produce the answer. This makes the behavior reproducible and illustrates the workflow, but it should not be described as actual LLM training.

The function `few_shot_prompting(model, examples, prompt)`:
1. Stores the demonstrations in the model.
2. Displays the examples as prompt context.
3. Calculates the target problem.
4. Returns a clear explanation and the final amount.

In [4]:
class SimpleModel:
    """A deterministic model simulation that stores examples and calculates investments."""

    def __init__(self):
        self.examples: List[Dict[str, Any]] = []

    def train(self, examples: List[Dict[str, Any]]) -> None:
        """Store few-shot demonstrations for use as contextual examples."""
        self.examples = list(examples)
        print(f"SimpleModel loaded with {len(self.examples)} worked example(s).")

    @staticmethod
    def predict(prompt: Dict[str, Any]) -> Dict[str, Any]:
        """Apply the explicit investment rules to a target problem."""
        validate_inputs(prompt)
        balance = float(prompt["initial_investment"])
        rows = []
        for year, rate in enumerate(prompt["annual_returns"], start=1):
            gain = balance * rate
            fee = float(prompt["annual_fee"])
            taxable_gain = max(0.0, gain - fee)
            tax = taxable_gain * float(prompt["tax_rate"])
            closing = balance + gain - fee - tax
            rows.append({
                "year": year,
                "opening_balance": balance,
                "return_rate": rate,
                "gain": gain,
                "fee": fee,
                "taxable_gain": taxable_gain,
                "tax": tax,
                "closing_balance": closing,
            })
            balance = closing
        return {"final_amount": balance, "yearly_log": rows}


def few_shot_prompting(
    model: SimpleModel,
    examples: List[Dict[str, Any]],
    prompt: Dict[str, Any]
) -> Dict[str, Any]:
    """Load examples, predict the target investment, and log a detailed explanation."""
    model.train(examples)

    print("\nFEW-SHOT DEMONSTRATIONS")
    for i, example in enumerate(model.examples, start=1):
        print(f"Example {i}:")
        print("  Input:", json.dumps(example["input"]))
        print(f"  Expected final amount: ${example['expected_final_amount']:,.2f}")

        # Verify the stored example's expected answer against the calculator.
        actual = model.predict(example["input"])["final_amount"]
        print(f"  Recalculated example  : ${actual:,.2f}")
        if abs(actual - example["expected_final_amount"]) > 0.01:
            print("  WARNING: expected example output does not match the calculation.")

    print("\nTARGET PROMPT")
    print(json.dumps(prompt, indent=2))
    prediction = model.predict(prompt)

    print("\nTARGET YEAR-BY-YEAR EXPLANATION")
    for row in prediction["yearly_log"]:
        print(
            f"Year {row['year']}: opening=${row['opening_balance']:,.2f}, "
            f"return={row['return_rate']:.2%}, gain=${row['gain']:,.2f}, "
            f"fee=${row['fee']:,.2f}, taxable gain=${row['taxable_gain']:,.2f}, "
            f"tax=${row['tax']:,.2f}, closing=${row['closing_balance']:,.2f}"
        )
    print(f"\nFew-shot predicted final amount: ${prediction['final_amount']:,.2f}")
    return prediction


# Small worked examples; outputs are computed and verified below.
example_1 = {
    "input": {
        "initial_investment": 1000.00,
        "annual_returns": [0.10],
        "annual_fee": 10.00,
        "tax_rate": 0.20,
    },
    # Gain=$100; net gain after fee=$90; tax=$18; final=$1,072.
    "expected_final_amount": 1072.00,
}
example_2 = {
    "input": {
        "initial_investment": 2000.00,
        "annual_returns": [-0.05],
        "annual_fee": 20.00,
        "tax_rate": 0.15,
    },
    # Loss=$100; no tax; final=$2,000-$100-$20=$1,880.
    "expected_final_amount": 1880.00,
}

simple_model = SimpleModel()
few_shot_result = few_shot_prompting(
    simple_model,
    examples=[example_1, example_2],
    prompt=INVESTMENT_INPUT
)

SimpleModel loaded with 2 worked example(s).

FEW-SHOT DEMONSTRATIONS
Example 1:
  Input: {"initial_investment": 1000.0, "annual_returns": [0.1], "annual_fee": 10.0, "tax_rate": 0.2}
  Expected final amount: $1,072.00
  Recalculated example  : $1,072.00
Example 2:
  Input: {"initial_investment": 2000.0, "annual_returns": [-0.05], "annual_fee": 20.0, "tax_rate": 0.15}
  Expected final amount: $1,880.00
  Recalculated example  : $1,880.00

TARGET PROMPT
{
  "initial_investment": 10000.0,
  "annual_returns": [
    0.08,
    -0.03,
    0.12,
    0.06,
    0.1
  ],
  "annual_fee": 100.0,
  "tax_rate": 0.2
}

TARGET YEAR-BY-YEAR EXPLANATION
Year 1: opening=$10,000.00, return=8.00%, gain=$800.00, fee=$100.00, taxable gain=$700.00, tax=$140.00, closing=$10,560.00
Year 2: opening=$10,560.00, return=-3.00%, gain=$-316.80, fee=$100.00, taxable gain=$0.00, tax=$0.00, closing=$10,143.20
Year 3: opening=$10,143.20, return=12.00%, gain=$1,217.18, fee=$100.00, taxable gain=$1,117.18, tax=$223.44, clos

## 4. Validation and comparison of all methods

This final check compares the final amounts from CoT, SC-CoT, and few-shot prompting. The methods should agree because they implement the same explicitly defined financial rules. Agreement verifies arithmetic consistency; it does not independently validate whether the assumed tax treatment matches a particular country's tax law.

In [5]:
results = {
    "Chain of Thought": cot_result["final_amount"],
    "Self-Consistency Method A": sc_result["method_a_final"],
    "Self-Consistency Method B": sc_result["method_b_final"],
    "Few-Shot Prompting": few_shot_result["final_amount"],
}

print("FINAL METHOD COMPARISON")
print("=" * 55)
for method, amount in results.items():
    print(f"{method:<30} ${amount:,.2f}")

amounts = list(results.values())
all_match = all(abs(amount - amounts[0]) <= 1e-8 for amount in amounts)
print("\nAll methods match:", all_match)
assert all_match, "A mismatch was detected between prompting-method implementations."
print("Validation passed: all methods returned the same final amount.")

FINAL METHOD COMPARISON
Chain of Thought               $12,325.66
Self-Consistency Method A      $12,325.66
Self-Consistency Method B      $12,325.66
Few-Shot Prompting             $12,325.66

All methods match: True
Validation passed: all methods returned the same final amount.


## 5. Summary and Conclusion

This assignment implemented three approaches to a five-year investment calculation with varying annual returns, a fixed annual fee, and tax on positive net annual gains.

- **Chain of Thought (CoT):** the year-by-year ledger makes the sequence of financial operations transparent and logs opening balance, return, fee, taxable gain, tax, and closing balance for each year.
- **Self-Consistency Chain of Thought (SC-CoT):** two independently written calculation paths are compared. Matching final amounts and yearly balances increase confidence that the arithmetic has been implemented consistently.
- **Few-Shot Prompting:** worked examples show the expected input and output pattern. The simulated `SimpleModel` stores and verifies examples, then produces a detailed answer for the target investment.
- **Accuracy and limitations:** all methods use the same stated assumptions, and the final assertion checks that their outputs match. This confirms implementation consistency, but it does not prove that the assumptions reflect a real investment product or tax jurisdiction. Actual financial calculations may differ based on fee timing, tax rules, loss carry-forward, compounding conventions, and rounding rules.

**Final result:** use the amount printed by the final comparison cell as the final investment value for the illustrative scenario. The key lesson is that explicit assumptions, step-by-step logging, independent calculation paths, and worked examples make financial problem-solving easier to audit and explain.